In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad, coordinate_offsets
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad

import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
hltau_c= SkyCoord("4h31m38.43s", "+18d13m57.19s", frame='fk5')
hltau_ref = hltau_c.skyoffset_frame()
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
distance_hltau = 147 #parsecs
distance_iras2a = 293 #parsecs
# choose which distance
distance = distance_iras2a

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'a
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
cubefile = 'test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
file_Tpeak = 'test_data/IRAS2A/D2CO_streamer_cluster_tpeak.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km


In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer

'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 6
vmax = 8
xmin = -5
xmax = 5
ymin = -12
ymax = 0.5
rms_thresh = 4

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 


In [ ]:
# optionally we can also plot the by eye streamline
add_by_eye = True
# by eye parameters
by_eye_params = {
    'r0': 2540.0,  # au
    'theta0': 54.0,  # degrees
    'phi0': 61.0,  # degrees
    'log_omega': np.log(7e-13),  # log(1/s)
    'v_r0': 0.0,  # km/s
}
fixed_params = {
    'mass': 4.0,  # solar masses
    'inc': -45.0,  # degrees
    'pa': 194.0,  # degrees
    'rmin': 50.0,  # au
    'deltar':50.0,  # au
    'v_lsr': 7.5  # km/s (systemic velocity)
}
# convert angles to radians for forward model
by_eye_params['theta0'] = np.radians(by_eye_params['theta0'])
by_eye_params['phi0'] = np.radians(by_eye_params['phi0'])
fixed_params['inc'] = np.radians(fixed_params['inc'])
fixed_params['pa'] = np.radians(fixed_params['pa'])
distance = 293
if add_by_eye:
    ra_by_eye, dec_by_eye, v_by_eye = gradient_descent.forward_model(by_eye_params, fixed_params, distance)

In [ ]:
n_points = 10 # the number of points we want to reduce the data to


# Extract 1D streamline from the data cube
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, n_elements=n_points)
print(f"point cloud velocities (km/s): {pc_coords[2]}")

# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]


data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

In [ ]:
# Helper function to compute and plot radial bin edges
def get_radial_bin_partitions(pc_coords, n_elements):
    """
    Compute the radial bin partition boundaries used in reduce_to_1D.
    
    Parameters
    ----------
    pc_coords : array of shape (3, n_points)
        Point cloud coordinates. Index 0 = RA, Index 1 = Dec, Index 2 = velocity
    n_elements : int
        Number of elements used in the reduction
    
    Returns
    -------
    partitions : array of shape (n_elements + 1,)
        Radial bin boundaries in arcsec
    """
    import numpy as np
    ra_coords = pc_coords[0]
    dec_coords = pc_coords[1]
    # Compute radial distance metric (same as in extract_streamline.get_distance_metric)
    distance_metric = np.sqrt(ra_coords**2 + dec_coords**2)
    # Compute percentile boundaries
    b_per = np.linspace(0, 100, n_elements + 1)
    partitions = np.array([np.percentile(distance_metric, per) for per in b_per])
    return partitions

def plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """
    Plot circles showing the radial bin edges on an RA-Dec plot.
    Axis limits are preserved after adding the circles.
    
    Parameters
    ----------
    ax : matplotlib axes object
        The axes to plot on
    partitions : array
        Radial bin boundaries in arcsec
    color : str
        Color of the circles
    linewidth : float
        Line width of the circles
    alpha : float
        Transparency of the circles
    """
    import matplotlib.patches as patches
    # Save current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Add circles
    for partition in partitions:
        circle = patches.Circle((0, 0), partition, fill=False, edgecolor=color, 
                               linewidth=linewidth, alpha=alpha)
        ax.add_patch(circle)
    
    # Restore original axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# plot it
import matplotlib.pyplot as plt

label_fs = 14
title_fs = 16
tick_fs = 12
cbar_fs = 12

fig, ax = plt.subplots(figsize=(6.5, 7))

# load image + WCS
tpeak_hdu = fits.open(file_Tpeak)[0]
tpeak_data = tpeak_hdu.data
tpeak_wcs = WCS(tpeak_hdu.header)

# replace zeros with NaNs for plotting
tpeak_data = np.where(tpeak_data == 0, np.nan, tpeak_data)

# pixel scale (arcsec/pixel)
pixscale_ra, pixscale_dec = [
    abs(s.to('deg').value * 3600)
    for s in tpeak_wcs.proj_plane_pixel_scales()
]

# star pixel coordinates
star_x, star_y = tpeak_wcs.world_to_pixel(iras2a_c)

# image extent in arcsec offsets from star
ny, nx = tpeak_data.shape
extent = [-(0-star_x)*pixscale_ra, -(nx-star_x)*pixscale_ra,
          (0-star_y)*pixscale_dec, (ny-star_y)*pixscale_dec]

# background image
'''
im = ax.imshow(
    tpeak_data,
    origin='lower',
    cmap='inferno',
    vmin=np.nanmin(tpeak_data),
    vmax=np.nanmax(tpeak_data),
    extent=extent
)


cbar = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.97)
cbar.set_label(r'T$_{peak}$ (K)', fontsize=label_fs)
cbar.ax.tick_params(labelsize=tick_fs)
'''
# background image (hidden but keeps layout space)
im = ax.imshow(
    np.zeros_like(tpeak_data),  # dummy data
    origin='lower',
    cmap='inferno',
    extent=extent,
    alpha=0  # invisible but still defines axes scaling
)

# colorbar placeholder (invisible but reserves space)
cbar = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.97)

# overlays
ax.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3,
           color='grey', label='Point cloud')

ax.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma,
            fmt='o-', color='red', label='Extracted 1D Streamline')

ax.scatter(0, 0, marker='*', s=100, color='yellow',
           edgecolor='black', zorder=10)

# optionally plot by-eye model
add_by_eye = False
if add_by_eye:
    plt.plot(ra_by_eye, dec_by_eye, color='tab:green', linewidth=2, label='By-eye streamline model', zorder=6)

# radial bins
partitions = get_radial_bin_partitions(pc_coords, n_points)
plot_radial_bin_circles(ax, partitions,
                        color='lightgrey', linewidth=1, alpha=0.3)

# formatting
ax.set_xlabel('RA Offset (arcsec)', fontsize=label_fs)
ax.set_ylabel('Dec Offset (arcsec)', fontsize=label_fs)
ax.set_aspect('equal')
# zoom
ax.set_xlim(5, -5)
ax.set_ylim(-12, 0.5)
ax.tick_params(axis='both', labelsize=tick_fs)
ax.legend(loc='lower right')
ax.set_title(r'Extracting 1D Streamline', fontsize=title_fs, pad=10)

In [ ]:
# first, if there is no velocity file made, make it
if not os.path.exists('test_data/IRAS2A/D2CO_streamer_cluster_velocity.fits'):
    # compute velocity file
    velocity_cube = streamer_cube.with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)
    velocity_data = velocity_cube.moment(order=1).value
    # make the header celestial WCS by dropping the spectral axis info
    velocity_cube.header['NAXIS'] = 2
    velocity_cube.header.pop('CRVAL3', None)
    velocity_cube.header.pop('CDELT3', None)
    velocity_cube.header.pop('CRPIX3', None)
    # save to fits
    vel_hdu = fits.PrimaryHDU(data=velocity_data, header=velocity_cube.header)
    vel_hdu.writeto('test_data/IRAS2A/D2CO_streamer_cluster_velocity.fits', overwrite=True)

# load it
vel_hdu = fits.open('test_data/IRAS2A/D2CO_streamer_cluster_velocity.fits')[0]
vel_map = vel_hdu.data
vel_wcs = WCS(vel_hdu.header).celestial


In [ ]:
# KDE of the velocity vs radial distance from the star
from scipy import stats
vlsr = 7.5

label_fs = 14
title_fs = 16
tick_fs = 12
legend_fs = 11

# --- KDE for velocity plot ---
## Choose limits for KDE
xmin, xmax = 0, 3500    #au
ymin, ymax = 6, 9      #km/s
# grid for kernel distribution
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])

# Create a proper 2D header from celestial WCS
vel_header_2d = vel_wcs.to_header()
vel_header_2d['NAXIS'] = 2
vel_header_2d['NAXIS1'] = vel_map.shape[1]
vel_header_2d['NAXIS2'] = vel_map.shape[0]

results = coordinate_offsets.generate_offsets(
    vel_header_2d,
    iras2a_c.ra,
    iras2a_c.dec,
    pa_angle=0*u.deg,
    inclination=0*u.deg
)
rproj_data = (results.r * distance * u.pc).to(u.au, equivalencies=u.dimensionless_angles())
vlos_data = vel_map * u.km/u.s

good = np.isfinite(rproj_data * vlos_data)
values = np.vstack([rproj_data[good].value, vlos_data[good].value])

# KDE calculation
kernel = stats.gaussian_kde(values)
zz = np.reshape(kernel(positions).T, xx.shape)
zz /= zz.max()
kde_levels = np.append(np.exp(-0.5 * np.arange(1.0, 2.1, 0.5)**2)[::-1], [1.0])

# now get the extracted streamline model values for plotting
rproj_model_arcsec = np.sqrt(ra_data**2 + dec_data**2) # arcsec
# Error propagation: σ_r = sqrt((ra * σ_ra / r)^2 + (dec * σ_dec / r)^2)
rproj_sigma_arcsec = np.sqrt((ra_data * ra_sigma / rproj_model_arcsec)**2 + 
                              (dec_data * dec_sigma / rproj_model_arcsec)**2)
# Convert to AU
rproj_model = (rproj_model_arcsec * u.arcsec * distance * u.pc).to(u.au, equivalencies=u.dimensionless_angles())
rproj_sigma = (rproj_sigma_arcsec * u.arcsec * distance * u.pc).to(u.au, equivalencies=u.dimensionless_angles())
vlos_model = v_data * u.km/u.s
print(f"model velocities (km/s): {vlos_model}")
print(f"model projected distances (au): {rproj_model}")
print(f"model projected distance errors (au): {rproj_sigma}")

# plot
fig, ax = plt.subplots(figsize=(6.5*1.3, 4*1.3))
ax.errorbar(rproj_model, vlos_model, xerr=rproj_sigma, yerr=v_sigma*u.km/u.s,
            fmt='o-', color='red', markersize=5, linewidth=1.8, elinewidth=1.1,
            label='Extracted 1D Streamline')
ax.contourf(xx, yy, zz, levels=kde_levels, cmap='Greys', vmin=0, vmax=1.2)
if add_by_eye:
    rproj_by_eye_arcsec = np.sqrt(ra_by_eye**2 + dec_by_eye**2) # arcsec
    rproj_by_eye = (rproj_by_eye_arcsec * u.arcsec * distance * u.pc).to(u.au, equivalencies=u.dimensionless_angles())
    vlos_by_eye = v_by_eye * u.km/u.s
    ax.plot(rproj_by_eye, vlos_by_eye, color='tab:green', linewidth=2, label='By-eye streamline model', zorder=6)
ax.axhline(vlsr, color='k', ls='--', lw=1.1)
ax.set_xlim((xmin, xmax))
ax.set_ylim((7.0, 7.8))
ax.set_xlabel('Projected Distance from Source (au)', fontsize=label_fs)
ax.set_ylabel(r'v$_{LOS}$ (km s$^{-1}$)', fontsize=label_fs)
ax.set_title(r'IRAS 2A D$_2$CO Streamer', fontsize=title_fs, pad=10)
ax.tick_params(axis='both', labelsize=tick_fs)
# add vertical lines for the radial bin edges
ax.vlines((partitions * u.arcsec * distance * u.pc).to(u.au, equivalencies=u.dimensionless_angles()).value,
          ymin, ymax, color='lightgrey', linestyle='-', alpha=0.3, zorder=0)
ax.legend(fontsize=legend_fs)